# Levine et al. (2015) -- Mass Cytometry of Human Bone Marrow (32-dim)

**Source:** Levine et al., "Data-Driven Phenotypic Dissection of AML Reveals Progenitor-like Cells that Correlate with Prognosis." *Cell* 162.1 (2015): 184--197.

265,627 human bone marrow cells profiled by mass cytometry (CyTOF) across 32 protein markers, with 14 manually gated cell populations (39% of cells labeled = 104,184 cells). CyTOF data features heavy-tailed protein expression distributions and overlapping cell populations with varying densities -- properties known to challenge classical distance-based validation indices (Silhouette, Davies--Bouldin, Calinski--Harabasz).

Available via the HDCytoData Bioconductor package (Weber & Robinson, 2016).

---

## 0. Imports & Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from benchmarks._studies import (
    STUDIES,
    carve_cache_path,
    cvi_sweep,
    fit_or_load_carve,
    study_model_grids,
)
from benchmarks.datasets import load_levine32
from benchmarks.figures import (
    figure_carve_output_levine,
    figure_levine_results,
    prepare_composite,
)

RANDOM_SEED = 42
study = STUDIES["levine32"]
model_grids = study_model_grids(study)


def show(fig):
    """Display a figure exactly once, then close it.

    A figure left open is rendered by the inline backend's end-of-cell
    flush, and a figure returned from a cell is rendered as that cell's
    result. A helper that does both renders twice in some cells and once in
    others. Displaying explicitly and closing is the one path that draws
    every figure exactly once.
    """
    display(fig)
    plt.close(fig)


# Manuscript line 627: a stratified subsample of 5,000 cells.
X, y, meta = load_levine32(subsample=5000, random_state=RANDOM_SEED)
print(f"{meta['n_cells']:,} cells x {meta['n_features']:,} genes")


---

## 1. Baseline Metrics

We evaluate Silhouette, Gap statistic, Davies-Bouldin, and Calinski-Harabasz across k = 7, ..., 17 for KMeans and spectral clustering (with self-tuning affinity):

In [ ]:
curves_df, best_df = cvi_sweep(
    X, y, model_grids=model_grids, candidate_k=study.candidate_k,
    random_state=RANDOM_SEED, n_jobs=-1,
)
best_df


---

## 2. CARVE Analysis

In [ ]:
carve = fit_or_load_carve(
    X, y,
    cache_path=carve_cache_path(study, root=Path("./carve_state_saves")),
    model_grids=model_grids,
    random_state=RANDOM_SEED,
)


In [ ]:
from sklearn.manifold import TSNE

tsne_embedding = TSNE(random_state=RANDOM_SEED).fit_transform(X)


---

## 3. Quantitative Comparison

### 3.1 Composite Paper Figure (with ARI Comparison)

In [ ]:
inputs = prepare_composite(
    X, y.to_numpy(), carve, curves_df=curves_df, best_df=best_df,
    comparison_metric="silhouette", embedding=tsne_embedding,
    measure="stability", rule="1se", not_two=True, random_state=RANDOM_SEED,
)

# Each figure writes itself under its manuscript filename in vis/
# case_studies/. Copying one into overleaf/vis/ stays a manual step.
show(figure_carve_output_levine(inputs))
show(figure_levine_results(inputs))


---

## 4. Summary

Levine_32dim is a canonical CyTOF benchmark with 14 manually gated bone marrow populations. The heavy-tailed protein expression distributions, overlapping populations, and varying population sizes challenge classical distance-based validation indices. We compare CARVE's stability-based selection (measure=s, rule=1se, not_two=True) against four classical metrics and quantify agreement with the 14 ground-truth populations using ARI.

---